# NSE Sectoral Rotation Project
## — Loading Data into PostgreSQL
Loading 3 master CSVs into nse_analysis database

In [14]:
import psycopg2
import pandas as pd

# Connect to PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    database="nse_analysis",
    user="postgres",
    password="postgres123"    # replace with your password
)

print("✅ Connected to PostgreSQL!")

✅ Connected to PostgreSQL!


## Step 1 - Load master_stocks.csv into PostgreSQL

In [15]:
# Load master stocks
master_stocks = pd.read_csv("../data/master_stocks.csv")
print(f"master_stocks shape: {master_stocks.shape}")
master_stocks.head()

master_stocks shape: (37467, 10)


,SYMBOL,OPEN,HIGH,LOW,CLOSE,PREVCLOSE,TOTTRDQTY,DATE,RETURN,SECTOR
0,ADANIGREEN,1935.55,1965.45,1907.20,1945.10,1914.70,1015269,2022-04-01,1.5877,ENERGY
1,ALKEM,3600.00,3619.85,3472.05,3493.20,3620.70,153314,2022-04-01,-3.5214,PHARMA
2,ASHOKLEY,117.40,119.70,116.30,119.15,117.25,12697606,2022-04-01,1.6205,AUTO
3,AUROPHARMA,673.85,684.45,663.00,682.35,668.55,1700773,2022-04-01,2.0642,PHARMA
4,AXISBANK,760.00,775.75,755.15,774.55,761.15,9321859,2022-04-01,1.7605,BANKING


## Step 2 - Create tables and load data

In [16]:
cur = conn.cursor()

# Create master_stocks table
cur.execute("""
    DROP TABLE IF EXISTS master_stocks;
    CREATE TABLE master_stocks (
        symbol VARCHAR(50),
        open NUMERIC,
        high NUMERIC,
        low NUMERIC,
        close NUMERIC,
        prevclose NUMERIC,
        tottrdqty BIGINT,
        date DATE,
        return NUMERIC,
        sector VARCHAR(50)
    );
""")
conn.commit()
print("✅ master_stocks table created!")

# Create sector_returns table
cur.execute("""
    DROP TABLE IF EXISTS sector_returns;
    CREATE TABLE sector_returns (
        sector VARCHAR(50),
        date DATE,
        avg_return NUMERIC
    );
""")
conn.commit()
print("✅ sector_returns table created!")

# Create index_returns table
cur.execute("""
    DROP TABLE IF EXISTS index_returns;
    CREATE TABLE index_returns (
        date DATE,
        nifty_close NUMERIC,
        nifty_return NUMERIC
    );
""")
conn.commit()
print("✅ index_returns table created!")

✅ master_stocks table created!
✅ sector_returns table created!
✅ index_returns table created!


## Step 3 - Insert CSV data into tables

In [17]:
import io

def load_csv_to_table(df, table_name, conn):
    buffer = io.StringIO()
    df.to_csv(buffer, index=False, header=False)
    buffer.seek(0)
    cur = conn.cursor()
    cur.copy_from(buffer, table_name, sep=",", null="")
    conn.commit()
    print(f"✅ {table_name} loaded — {len(df):,} rows")

# Load all 3 CSVs
master_stocks = pd.read_csv("../data/master_stocks.csv")
sector_returns = pd.read_csv("../data/sector_returns.csv")
index_returns = pd.read_csv("../data/index_returns.csv")

load_csv_to_table(master_stocks, "master_stocks", conn)
load_csv_to_table(sector_returns, "sector_returns", conn)
load_csv_to_table(index_returns, "index_returns", conn)

✅ master_stocks loaded — 37,467 rows
✅ sector_returns loaded — 4,944 rows
✅ index_returns loaded — 737 rows


## Step 4 - Verify data loaded correctly

In [18]:
cur = conn.cursor()

# Check row counts in each table
cur.execute("SELECT COUNT(*) FROM master_stocks;")
print(f"master_stocks rows: {cur.fetchone()[0]:,}")

cur.execute("SELECT COUNT(*) FROM sector_returns;")
print(f"sector_returns rows: {cur.fetchone()[0]:,}")

cur.execute("SELECT COUNT(*) FROM index_returns;")
print(f"index_returns rows: {cur.fetchone()[0]:,}")

print("\n✅ All data loaded into PostgreSQL successfully!")

master_stocks rows: 37,467
sector_returns rows: 4,944
index_returns rows: 737

✅ All data loaded into PostgreSQL successfully!


## Step 5 - Preview each table

In [19]:
# Preview master_stocks
print("=== master_stocks ===")
cur.execute("SELECT * FROM master_stocks LIMIT 5;")
print(pd.DataFrame(cur.fetchall()).to_string())

# Preview sector_returns
print("\n=== sector_returns ===")
cur.execute("SELECT * FROM sector_returns LIMIT 5;")
print(pd.DataFrame(cur.fetchall()).to_string())

# Preview index_returns
print("\n=== index_returns ===")
cur.execute("SELECT * FROM index_returns LIMIT 5;")
print(pd.DataFrame(cur.fetchall()).to_string())

=== master_stocks ===
            0        1        2        3       4       5         6           7        8        9
0  ADANIGREEN  1935.55  1965.45   1907.2  1945.1  1914.7   1015269  2022-04-01   1.5877   ENERGY
1       ALKEM   3600.0  3619.85  3472.05  3493.2  3620.7    153314  2022-04-01  -3.5214   PHARMA
2    ASHOKLEY    117.4    119.7    116.3  119.15  117.25  12697606  2022-04-01   1.6205     AUTO
3  AUROPHARMA   673.85   684.45    663.0  682.35  668.55   1700773  2022-04-01   2.0642   PHARMA
4    AXISBANK    760.0   775.75   755.15  774.55  761.15   9321859  2022-04-01   1.7605  BANKING

=== sector_returns ===
      0           1        2
0  AUTO  2022-01-03   1.8217
1  AUTO  2022-01-04   0.0252
2  AUTO  2022-01-05   1.6571
3  AUTO  2022-01-06   0.7654
4  AUTO  2022-01-07  -0.2811

=== index_returns ===
            0                1        2
0  2022-01-04         17805.25   1.0187
1  2022-01-05         17925.25    0.674
2  2022-01-06  17745.900390625  -1.0005
3  2022-01-07  

## Step 6 - Close connection

In [20]:
cur.close()
conn.close()
print("✅ Connection closed!")



✅ Connection closed!
